<a href="https://colab.research.google.com/github/Ariiiiii22/MetroCDMX_DFS-y-BFS/blob/main/Lineas_del_metro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from collections import deque

# Clase abstracta
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial # Estado inicial
        self.goal = goal # Meta

    def actions(self, state):
        raise NotImplementedError

    # Función de transición
    def result(self, state, action):
        raise NotImplementedError

    # Función de desempeño
    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0


class GraphProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        return list(self.graph[state].keys())

    # Función de transición
    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

In [4]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action if action else state
        self.path_cost = path_cost

    def __repr__(self):
        return f"<Node state:{self.state}, path_cost:{self.path_cost}>"

    def solution(self):
        return " -> ".join([node.action for node in self.path()])

    def path(self):
        node, path_back = self, []
        while node:
            path_back.append(node)
            node = node.parent
        return list(reversed(path_back))

    def expand(self, problem):
        return [self.child_node(problem, action) for action in problem.actions(self.state)]

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

In [5]:
def depth_first_graph_search(problem):
    """DFS: frontera = pila (LIFO), explora el nodo más profundo primero."""
    start_node = Node(problem.initial)
    if problem.is_goal(start_node.state):
        return start_node

    frontier = [start_node]
    explored = set()
    frontier_states = {start_node.state}

    while frontier:
        node = frontier.pop()                  # LIFO
        frontier_states.remove(node.state)
        explored.add(node.state)

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in explored and child.state not in frontier_states:
                frontier.append(child)
                frontier_states.add(child.state)
    return None


def breadth_first_graph_search(problem):
    """BFS: frontera = cola (FIFO), explora por niveles."""
    start_node = Node(problem.initial)
    if problem.is_goal(start_node.state):
        return start_node

    frontier = deque([start_node])
    explored = set()
    frontier_states = {start_node.state}

    while frontier:
        node = frontier.popleft()              # FIFO
        frontier_states.remove(node.state)
        explored.add(node.state)

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in explored and child.state not in frontier_states:
                frontier.append(child)
                frontier_states.add(child.state)
    return None

In [6]:
LINEAS = {
    "1": ["Observatorio", "Tacubaya", "Juanacatlán", "Chapultepec", "Sevilla", "Insurgentes",
          "Cuauhtémoc", "Balderas", "Salto del Agua", "Isabel la Católica", "Pino Suárez", "Merced",
          "Candelaria", "San Lázaro", "Moctezuma", "Balbuena", "Boulevard Puerto Aéreo",
          "Gómez Farías", "Zaragoza", "Pantitlán"],
    "2": ["Cuatro Caminos", "Panteones", "Tacuba", "Cuitláhuac", "Popotla", "Colegio Militar",
          "Normal", "San Cosme", "Revolución", "Hidalgo", "Bellas Artes", "Allende", "Zócalo",
          "Pino Suárez", "San Antonio Abad", "Chabacano", "Viaducto", "Xola", "Villa de Cortés",
          "Nativitas", "Portales", "Ermita", "General Anaya", "Tasqueña"],
    "3": ["Indios Verdes", "Deportivo 18 de Marzo", "Potrero", "La Raza", "Tlatelolco", "Guerrero",
          "Hidalgo", "Juárez", "Balderas", "Niños Héroes", "Hospital General", "Centro Médico",
          "Etiopía", "Eugenia", "División del Norte", "Zapata", "Coyoacán", "Viveros",
          "Miguel Ángel de Quevedo", "Copilco", "Universidad"],
    "4": ["Martín Carrera", "Talismán", "Bondojito", "Consulado", "Canal del Norte", "Morelos",
          "Candelaria", "Fray Servando", "Jamaica", "Santa Anita"],
    "5": ["Pantitlán", "Hangares", "Terminal Aérea", "Oceanía", "Aragón", "Eduardo Molina",
          "Consulado", "Valle Gómez", "Misterios", "La Raza", "Autobuses del Norte",
          "Instituto del Petróleo", "Politécnico"],
    "6": ["El Rosario", "Tezozómoc", "UAM-Azcapotzalco", "Ferrería", "Norte 45", "Vallejo",
          "Instituto del Petróleo", "Lindavista", "Deportivo 18 de Marzo", "La Villa-Basílica",
          "Martín Carrera"],
    "7": ["El Rosario", "Aquiles Serdán", "Camarones", "Refinería", "Tacuba", "San Joaquín",
          "Polanco", "Auditorio", "Constituyentes", "Tacubaya", "San Pedro de los Pinos",
          "San Antonio", "Mixcoac", "Barranca del Muerto"],
    "8": ["Garibaldi", "Bellas Artes", "San Juan de Letrán", "Salto del Agua", "Doctores", "Obrera",
          "Chabacano", "La Viga", "Santa Anita", "Coyuya", "Iztacalco", "Apatlaco", "Aculco",
          "Escuadrón 201", "Atlalilco", "Iztapalapa", "Cerro de la Estrella", "UAM-I",
          "Constitución de 1917"],
    "9": ["Tacubaya", "Patriotismo", "Chilpancingo", "Centro Médico", "Lázaro Cárdenas", "Chabacano",
          "Jamaica", "Mixiuhca", "Velódromo", "Ciudad Deportiva", "Puebla", "Pantitlán"],
    "A": ["Pantitlán", "Agrícola Oriental", "Canal de San Juan", "Tepalcates", "Guelatao",
          "Peñón Viejo", "Acatitla", "Santa Marta", "Los Reyes", "La Paz"],
    "B": ["Buenavista", "Guerrero", "Garibaldi", "Lagunilla", "Tepito", "Morelos", "San Lázaro",
          "Ricardo Flores Magón", "Romero Rubio", "Oceanía", "Deportivo Oceanía", "Bosque de Aragón",
          "Villa de Aragón", "Nezahualcóyotl", "Impulsora", "Río de los Remedios", "Múzquiz",
          "Ecatepec", "Olímpica", "Plaza Aragón", "Ciudad Azteca"],
    "12": ["Mixcoac", "Insurgentes Sur", "Hospital 20 de Noviembre", "Zapata", "Parque de los Venados",
           "Eje Central", "Ermita", "Mexicaltzingo", "Atlalilco", "Culhuacán", "San Andrés Tomatlán",
           "Lomas Estrella", "Calle 11", "Periférico Oriente", "Tezonco", "Olivos", "Nopalera",
           "Zapotitlán", "Tlaltenco", "Tláhuac"],
}

In [7]:
def construir_metro(lineas):
    """Devuelve un dict con el mismo formato que romania: {estacion: {vecino: costo}}."""
    grafo = {}
    for estaciones in lineas.values():
        for e in estaciones:
            grafo.setdefault(e, {})
        for a, b in zip(estaciones, estaciones[1:]):
            grafo[a][b] = 1          # costo unitario
            grafo[b][a] = 1
    return grafo

metro = construir_metro(LINEAS)
print("Estaciones:", len(metro))
print("Tramos:", sum(len(v) for v in metro.values()) // 2)
print("Vecinos de Pantitlán:", metro["Pantitlán"])

Estaciones: 163
Tramos: 183
Vecinos de Pantitlán: {'Zaragoza': 1, 'Hangares': 1, 'Puebla': 1, 'Agrícola Oriental': 1}


In [8]:
rutas = [("Cuatro Caminos", "Pantitlán"),
         ("Politécnico", "Tasqueña"),
         ("Zapata", "Oceanía")]

resultados = {}
for inicio, meta in rutas:
    problem = GraphProblem(inicio, meta, metro)
    n_dfs = depth_first_graph_search(problem)
    n_bfs = breadth_first_graph_search(problem)
    resultados[(inicio, meta)] = (n_dfs, n_bfs)

    print("=" * 70)
    print(f"{inicio} -> {meta}")
    print("=" * 70)
    print(f"DFS ({int(n_dfs.path_cost)} pasos):")
    print("  " + n_dfs.solution())
    print(f"BFS ({int(n_bfs.path_cost)} pasos):")
    print("  " + n_bfs.solution())
    print()

Cuatro Caminos -> Pantitlán
DFS (18 pasos):
  Cuatro Caminos -> Panteones -> Tacuba -> San Joaquín -> Polanco -> Auditorio -> Constituyentes -> Tacubaya -> Patriotismo -> Chilpancingo -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Mixiuhca -> Velódromo -> Ciudad Deportiva -> Puebla -> Pantitlán
BFS (18 pasos):
  Cuatro Caminos -> Panteones -> Tacuba -> San Joaquín -> Polanco -> Auditorio -> Constituyentes -> Tacubaya -> Patriotismo -> Chilpancingo -> Centro Médico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Mixiuhca -> Velódromo -> Ciudad Deportiva -> Puebla -> Pantitlán

Politécnico -> Tasqueña
DFS (46 pasos):
  Politécnico -> Instituto del Petróleo -> Lindavista -> Deportivo 18 de Marzo -> La Villa-Basílica -> Martín Carrera -> Talismán -> Bondojito -> Consulado -> Valle Gómez -> Misterios -> La Raza -> Tlatelolco -> Guerrero -> Garibaldi -> Lagunilla -> Tepito -> Morelos -> San Lázaro -> Ricardo Flores Magón -> Romero Rubio -> Oceanía -> Terminal Aérea -> Hangares 